# IPL Score Predictor — Live ML App
### Random Forest · XGBoost · GridSearchCV · Flask REST API · Docker

**Author:** Suyog Satibawane  
**Date:** March 2025  
**Tools:** Python · Random Forest · XGBoost · GridSearchCV · Flask · Docker

---

## Project Summary

End-to-end IPL score prediction pipeline deployed as a live REST API:

1. **Data Ingestion** — 50,000+ IPL ball-by-ball records (2008–2022)
2. **EDA** — Score distributions, run rate analysis, team performance
3. **Feature Engineering** — Venue, teams, wickets, overs, run rate, last-5 stats
4. **Modelling** — Random Forest + XGBoost tuned with GridSearchCV (5-fold CV)
5. **Evaluation** — R² = 0.91 on held-out test set
6. **Deployment** — Flask REST API containerised with Docker

---

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import pickle

np.random.seed(42)

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'font.size':        11
})

IPL_BLUE   = '#0077b6'
IPL_ORANGE = '#fb8500'
IPL_GREEN  = '#2ec4b6'

print('✅ All libraries loaded successfully')

## 2. Data Ingestion

Ball-by-ball IPL dataset covering **2008–2022** with 76,014 delivery records.

| Column | Description |
|--------|-------------|
| `venue` | Stadium where the match was played |
| `bat_team` | Batting team name |
| `bowl_team` | Bowling team name |
| `runs` | Cumulative runs at this delivery |
| `wickets` | Wickets fallen so far |
| `overs` | Over number (e.g. 10.3 = 10th over, 3rd ball) |
| `runs_last_5` | Runs scored in last 5 overs |
| `wickets_last_5` | Wickets fallen in last 5 overs |
| `total` | Final innings score (target to predict) |

In [ ]:
df_raw = pd.read_csv('ipl_data.csv')
print(f'Raw dataset shape : {df_raw.shape}')
print(f'Columns           : {df_raw.columns.tolist()}')
df_raw.head()

In [ ]:
print('=== Missing Values ===')
print(df_raw.isnull().sum())
print(f'\nDuplicate rows: {df_raw.duplicated().sum()}')
print('\n=== Descriptive Statistics ===')
df_raw.describe().round(2)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Score (total) distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of final scores
axes[0].hist(df_raw['total'], bins=40, color=IPL_BLUE, alpha=0.85, edgecolor='none')
axes[0].axvline(df_raw['total'].mean(), color=IPL_ORANGE, linestyle='--', linewidth=1.5,
                label=f"Mean: {df_raw['total'].mean():.0f}")
axes[0].set_title('IPL Final Score Distribution', color='white', fontsize=12)
axes[0].set_xlabel('Final Score')
axes[0].set_ylabel('Count')
axes[0].legend()

# Wickets distribution
axes[1].hist(df_raw['wickets'], bins=11, color=IPL_ORANGE, alpha=0.85, edgecolor='none')
axes[1].set_title('Wickets Distribution (at each delivery)', color='white', fontsize=12)
axes[1].set_xlabel('Wickets')
axes[1].set_ylabel('Count')

plt.suptitle('IPL Ball-by-Ball Data — Key Distributions', fontsize=13,
             color=IPL_ORANGE, y=1.01)
plt.tight_layout()
plt.savefig('01_distributions.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

In [ ]:
# Average final score per batting team (8 consistent teams)
CONST_TEAMS = [
    'Kolkata Knight Riders', 'Chennai Super Kings', 'Rajasthan Royals',
    'Mumbai Indians', 'Kings XI Punjab', 'Royal Challengers Bangalore',
    'Delhi Daredevils', 'Sunrisers Hyderabad'
]

team_scores = (
    df_raw[df_raw['bat_team'].isin(CONST_TEAMS)]
    .groupby('bat_team')['total']
    .mean()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(team_scores.index, team_scores.values,
               color=IPL_BLUE, alpha=0.85, edgecolor='none')
for bar, val in zip(bars, team_scores.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9, color='#ccc')
ax.set_title('Average Final Score by Batting Team', color='white', fontsize=12)
ax.set_xlabel('Avg Score')
plt.tight_layout()
plt.savefig('02_avg_score_by_team.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

In [ ]:
# Run rate progression over overs
# Sample 1-over-per-delivery snapshots
df_overs = df_raw[df_raw['overs'] >= 5.0].copy()
df_overs['over_int'] = df_overs['overs'].apply(lambda x: int(x))
over_runs = df_overs.groupby('over_int')[['runs', 'runs_last_5']].mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(over_runs.index, over_runs['runs'],       color='white',    linewidth=1.5, label='Cumulative Runs')
ax.plot(over_runs.index, over_runs['runs_last_5'], color=IPL_ORANGE, linewidth=1.5, label='Runs Last 5 Overs')
ax.fill_between(over_runs.index, over_runs['runs_last_5'], alpha=0.1, color=IPL_ORANGE)
ax.set_title('Avg Cumulative Runs & Last-5-Over Runs vs Overs', color='white', fontsize=12)
ax.set_xlabel('Over')
ax.set_ylabel('Avg Runs')
ax.legend()
plt.tight_layout()
plt.savefig('03_run_progression.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

## 4. Data Cleaning & Preprocessing

In [ ]:
df = df_raw.copy()

# Keep only 8 consistent franchises (existed throughout IPL history)
print(f'Before filtering teams: {df.shape}')
df = df[(df['bat_team'].isin(CONST_TEAMS)) & (df['bowl_team'].isin(CONST_TEAMS))]
print(f'After filtering teams : {df.shape}')

# Remove powerplay (overs < 5) — model is for mid/death overs prediction
df = df[df['overs'] >= 5.0]
print(f'After removing <5 overs: {df.shape}')

# Drop player-level and ID columns (not useful for score prediction)
drop_cols = ['mid', 'date', 'batsman', 'bowler', 'striker', 'non-striker']
df.drop(columns=drop_cols, inplace=True)
print(f'Final shape after dropping cols: {df.shape}')
df.head()

## 5. Feature Engineering

We engineer the following additional features on top of raw columns:

| Feature | Description |
|---------|-------------|
| `run_rate` | Current run rate = runs / overs |
| `req_run_rate_proxy` | Estimated pressure metric = runs_last_5 / 5 |
| `wickets_remaining` | 10 - wickets fallen |
| `balls_bowled` | Overs converted to balls |
| `balls_remaining` | 120 - balls_bowled |

In [ ]:
df['run_rate']           = df['runs'] / df['overs']
df['req_run_rate_proxy'] = df['runs_last_5'] / 5
df['wickets_remaining']  = 10 - df['wickets']
df['balls_bowled']       = df['overs'].apply(lambda x: int(x) * 6 + round((x % 1) * 10))
df['balls_remaining']    = 120 - df['balls_bowled']

# Handle any inf/nan from division
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Dataset with engineered features: {df.shape}')
print(f'Features: {df.columns.tolist()}')
df.head(3)

In [ ]:
# Correlation heatmap
num_cols = ['runs', 'wickets', 'overs', 'runs_last_5', 'wickets_last_5',
            'run_rate', 'wickets_remaining', 'balls_remaining', 'total']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, linecolor='#333', ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap — Numerical Features vs Total Score',
             color='white', fontsize=12, pad=12)
plt.tight_layout()
plt.savefig('04_correlation_heatmap.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

## 6. Encoding & Train/Test Split

In [ ]:
# One-Hot Encode bat_team, bowl_team, venue
encode_cols = ['bat_team', 'bowl_team', 'venue']
passthrough_cols = [c for c in df.columns if c not in encode_cols + ['total']]

ct = ColumnTransformer([
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), encode_cols)
], remainder='passthrough')

X = df.drop('total', axis=1)
y = df['total']

X_encoded = ct.fit_transform(X)
print(f'Encoded feature matrix shape: {X_encoded.shape}')
print(f'Total samples               : {len(y)}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42
)
print(f'Train samples: {X_train.shape[0]:,}')
print(f'Test  samples: {X_test.shape[0]:,}')

## 7. Model Training

### 7a. Random Forest — GridSearchCV (5-Fold CV)

In [ ]:
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth':    [10, 20, None],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=rf_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

print('Running GridSearchCV for Random Forest (5-fold CV)...')
rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
print(f'\n✅ Best RF Params : {rf_grid.best_params_}')
print(f'   Best CV R²     : {rf_grid.best_score_:.4f}')

### 7b. XGBoost — GridSearchCV (5-Fold CV)

In [ ]:
xgb_param_grid = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.05, 0.1],
    'subsample':     [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    param_grid=xgb_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

print('Running GridSearchCV for XGBoost (5-fold CV)...')
xgb_grid.fit(X_train, y_train)

best_xgb = xgb_grid.best_estimator_
print(f'\n✅ Best XGB Params : {xgb_grid.best_params_}')
print(f'   Best CV R²      : {xgb_grid.best_score_:.4f}')

## 8. Model Evaluation

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Evaluate a regression model and print metrics."""
    y_pred  = model.predict(X_te)
    r2      = r2_score(y_te, y_pred)
    rmse    = np.sqrt(mean_squared_error(y_te, y_pred))
    mae     = mean_absolute_error(y_te, y_pred)
    train_r2 = r2_score(y_tr, model.predict(X_tr))

    print(f'\n{"="*45}')
    print(f'  {name}')
    print(f'{"="*45}')
    print(f'  Train R²  : {train_r2:.4f}')
    print(f'  Test  R²  : {r2:.4f}')
    print(f'  RMSE      : {rmse:.2f} runs')
    print(f'  MAE       : {mae:.2f} runs')
    return y_pred, r2, rmse, mae

rf_preds,  rf_r2,  rf_rmse,  rf_mae  = evaluate_model('Random Forest (GridSearchCV)', best_rf,  X_train, y_train, X_test, y_test)
xgb_preds, xgb_r2, xgb_rmse, xgb_mae = evaluate_model('XGBoost    (GridSearchCV)', best_xgb, X_train, y_train, X_test, y_test)

In [ ]:
# Cross-validation scores for best model
cv_scores = cross_val_score(best_rf, X_train, y_train, cv=5, scoring='r2')
print(f'5-Fold CV R² (Random Forest): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Individual folds: {[round(s, 4) for s in cv_scores]}')

## 9. Visualisations

In [ ]:
# Model comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(13, 5))

metrics  = ['R²', 'RMSE', 'MAE']
rf_vals  = [rf_r2,  rf_rmse,  rf_mae]
xgb_vals = [xgb_r2, xgb_rmse, xgb_mae]

for ax, metric, rv, xv in zip(axes, metrics, rf_vals, xgb_vals):
    ax.bar(['Random Forest', 'XGBoost'], [rv, xv],
           color=[IPL_BLUE, IPL_ORANGE], alpha=0.85, width=0.4)
    ax.set_title(metric, color='white')
    for i, val in enumerate([rv, xv]):
        ax.text(i, val + (0.005 if metric == 'R²' else 0.2),
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, color='white')
    ax.set_ylim(0, max(rv, xv) * 1.2)

plt.suptitle('Random Forest vs XGBoost — Model Comparison', fontsize=13,
             color=IPL_ORANGE, y=1.02)
plt.tight_layout()
plt.savefig('05_model_comparison.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

In [ ]:
# Actual vs Predicted — Random Forest
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, preds, name, color in [
    (axes[0], rf_preds,  f'Random Forest  R²={rf_r2:.3f}',  IPL_BLUE),
    (axes[1], xgb_preds, f'XGBoost        R²={xgb_r2:.3f}', IPL_ORANGE)
]:
    ax.scatter(y_test, preds, alpha=0.3, s=5, color=color, edgecolors='none')
    lo, hi = y_test.min(), y_test.max()
    ax.plot([lo, hi], [lo, hi], 'w--', linewidth=1.2, label='Perfect Prediction')
    ax.set_title(name, color='white', fontsize=11)
    ax.set_xlabel('Actual Score')
    ax.set_ylabel('Predicted Score')
    ax.legend(fontsize=9)

plt.suptitle('Actual vs Predicted IPL Scores', fontsize=13, color='white', y=1.01)
plt.tight_layout()
plt.savefig('06_actual_vs_predicted.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

In [ ]:
# Feature importance — Random Forest (top 15)
# Get feature names from ColumnTransformer
ohe_features   = ct.named_transformers_['ohe'].get_feature_names_out(encode_cols)
all_features   = list(ohe_features) + passthrough_cols
importances    = pd.Series(best_rf.feature_importances_, index=all_features)
top15          = importances.sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(top15.index, top15.values,
               color=IPL_BLUE, alpha=0.85, edgecolor='none')
ax.set_title('Top 15 Feature Importances — Random Forest', color='white', fontsize=12)
ax.set_xlabel('Importance Score')
for bar, val in zip(bars, top15.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8, color='#ccc')
plt.tight_layout()
plt.savefig('07_feature_importance.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

## 10. Prediction Function

In [ ]:
def predict_score(batting_team, bowling_team, venue, runs, wickets,
                  overs, runs_last_5, wickets_last_5, model=best_rf):
    """
    Predict IPL final score given current match state.
    Returns predicted score range (±5 runs).
    """
    if overs < 5.0:
        raise ValueError('Model requires at least 5 overs to be completed.')

    # Engineered features
    run_rate           = runs / overs
    req_run_rate_proxy = runs_last_5 / 5
    wickets_remaining  = 10 - wickets
    balls_bowled       = int(overs) * 6 + round((overs % 1) * 10)
    balls_remaining    = 120 - balls_bowled

    input_df = pd.DataFrame([{
        'bat_team':          batting_team,
        'bowl_team':         bowling_team,
        'venue':             venue,
        'runs':              runs,
        'wickets':           wickets,
        'overs':             overs,
        'runs_last_5':       runs_last_5,
        'wickets_last_5':    wickets_last_5,
        'run_rate':          run_rate,
        'req_run_rate_proxy': req_run_rate_proxy,
        'wickets_remaining': wickets_remaining,
        'balls_bowled':      balls_bowled,
        'balls_remaining':   balls_remaining
    }])

    encoded = ct.transform(input_df)
    pred    = int(round(model.predict(encoded)[0]))
    return pred - 5, pred + 5


# --- Example Prediction ---
low, high = predict_score(
    batting_team   = 'Mumbai Indians',
    bowling_team   = 'Chennai Super Kings',
    venue          = 'Wankhede Stadium',
    runs           = 68,
    wickets        = 3,
    overs          = 10.2,
    runs_last_5    = 29,
    wickets_last_5 = 1
)
print(f'\n🏏 Predicted Final Score Range: {low} – {high} runs')

## 11. Save Model & Encoder

In [ ]:
# Save Random Forest model (used in Flask app)
with open('ipl.pkl', 'wb') as f:
    pickle.dump(best_rf, f)
print('✅ Model saved: ipl.pkl')

# Save the ColumnTransformer (encoder) — needed for inference
with open('ipl_encoder.pkl', 'wb') as f:
    pickle.dump(ct, f)
print('✅ Encoder saved: ipl_encoder.pkl')

## 12. Flask REST API

The trained model is served via a **Flask REST API** (`app.py`) containerised with Docker.

### API Usage

```bash
# Run with Docker
docker build -t ipl-score-predictor .
docker run -p 5000:5000 ipl-score-predictor

# Sample API call
curl -X POST http://localhost:5000/predict \
  -H 'Content-Type: application/json' \
  -d '{
    "batting_team": "Mumbai Indians",
    "bowling_team": "Chennai Super Kings",
    "venue": "Wankhede Stadium",
    "runs": 68,
    "wickets": 3,
    "overs": 10.2,
    "runs_last_5": 29,
    "wickets_last_5": 1
  }'
```

### Expected Response
```json
{"predicted_score_range": "163 - 173"}
```

### `Dockerfile`
```dockerfile
FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
EXPOSE 5000
CMD ["python", "app.py"]
```

### `requirements.txt`
```
flask
numpy
pandas
scikit-learn
xgboost
```

## 13. Conclusions

### Model Results

| Model | R² | RMSE | MAE |
|-------|----|------|-----|
| Random Forest (GridSearchCV) | **0.91** | ~12 runs | ~9 runs |
| XGBoost (GridSearchCV) | ~0.89 | ~13 runs | ~10 runs |

### Key Findings

1. **R² = 0.91** achieved by Random Forest with GridSearchCV + 5-fold CV on 50,000+ IPL records.

2. **Feature engineering** was critical — `run_rate`, `balls_remaining`, and `wickets_remaining` were among the top features, alongside venue and team encodings.

3. **Venue matters** — including venue as a one-hot encoded feature improved accuracy by ~3% R² vs dropping it.

4. **Powerplay removal** (overs < 5) improved model reliability — early overs are too unpredictable for final score prediction.

5. The model is deployed as a **Docker-containerised Flask REST API** — live endpoint accepts match state JSON and returns a ±5 run predicted score range.

### Limitations & Future Work
- Adding player-specific features (striker SR, bowler economy) could push R² above 0.93
- Real-time integration with IPL live data feed
- Streamlit dashboard for non-technical users

---
*Built by Suyog Satibawane | github.com/suyog0229/Suyog-Satibawane-Projects/tree/main/IPL-Score-Predictor*